# 06｜事件、状态与消息


> 运行方式：从上到下执行代码单元。控制类 Notebook **没有布尔安全开关**；执行控制单元就会把任务提交给 SDK。
> 需要仿真的课程请先在 FleetDock IDEA 中启动仿真，并确认 `drone1`、`drone2` 等设备在线。


![事件、状态与标准传感器](assets/events-sensors.svg)


## 强类型事件

适配器把设备遥测转换成标准事件。应用按 `EventType` 订阅，不需要解析 MAVLink 消息。


In [ ]:
import time
from fleetdock_runtime import FleetDock
from fleetdock import EventType

FleetDock.init()
drone = FleetDock.devices.drone1

unsubscribe = drone.events.on(
    EventType.GLOBAL_POSITION_UPDATED,
    lambda event: print("全球位置：", event.position),
)

print(drone.connect())
time.sleep(5.0)
unsubscribe()


## 状态快照

`device.state` 会持续汇总标准事件。需要当前状态时直接取得不可变快照，不要自己维护全局变量。


In [ ]:
snapshot = drone.state.snapshot()
print("连接：", snapshot.connected)
print("解锁：", snapshot.armed)
print("飞行中：", snapshot.in_air)
print("本地位置：", snapshot.local_x_m, snapshot.local_y_m, snapshot.local_z_m)
print("剩余电量：", snapshot.battery_remaining_fraction)


## 应用内消息

`EventHub` 提供轻量的进程内消息发布与订阅，适合把任务进度交给界面或其它业务模块。


In [ ]:
from fleetdock import EventHub, MessageEventType

hub = EventHub()
unsubscribe_message = hub.subscribe(
    MessageEventType.CUSTOM,
    lambda event: print("应用消息：", event.topic, event.payload),
    topic="lesson.progress",
)
hub.publish_custom("lesson.progress", {"step": 1, "message": "事件课程完成"})
unsubscribe_message()


In [ ]:
FleetDock.close()


## 重点

`events` 适合观察变化，`state.snapshot()` 适合读取最新状态。两者使用同一套 SDK 标准模型。
